# 1. 단어 사전 기반 매핑 복원

## 1) Import 

In [3]:
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 28.2 MB/s eta 0:00:00:00:0100:01


In [5]:
import pandas as pd

## 2) Data Load

In [6]:
train = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/train.csv', encoding = 'utf-8')
test = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/test.csv', encoding = 'utf-8')

## 3) 단어 사전 생성

In [7]:
match_dict = {}

for input_text, output_text in zip(train['input'], train['output']):
    input_words = input_text.split()
    output_words = output_text.split()
    for iw, ow in zip(input_words, output_words):
        match_dict[iw] = ow  

## 4) 변환 적용

In [8]:
def replace_words(input_text, match_dict):
    words = input_text.split() 
    replaced_words = [match_dict.get(word, word) for word in words] 
    return " ".join(replaced_words)

In [10]:
converted_reviews = test['input'].apply(lambda x: replace_words(x, match_dict)).tolist()

## 5) Submission

In [11]:
submission = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/sample_submission.csv', encoding = 'utf-8')

In [9]:
submission['output'] = converted_reviews

In [10]:
submission.to_csv('./submission.csv', index = False, encoding = 'utf-8')

# 2. LLM 활용 (Gemma)

## 1) Import 

In [13]:
import torch 
from transformers import AutoTokenizer, pipeline

## 2) Data Load

In [14]:
train = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/train.csv', encoding = 'utf-8')
test = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/test.csv', encoding = 'utf-8')

In [16]:
train = train[:3000]
test = test[:200]

In [17]:
print(train)

               ID                                              input  \
0     TRAIN_00000  별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈...   
1     TRAIN_00001                             잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ   
2     TRAIN_00002                                    절테 간면 않 된는 굣 멥몫   
3     TRAIN_00003  야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵...   
4     TRAIN_00004  집윈 축쳐눌료 딴너왓눈뎁 카셩뷔 좋곱 칼쿰한네올. 쩌럼한뒈 뮬콰 욺료토 잊쿄 빻토 ...   
...           ...                                                ...   
2995  TRAIN_02995                     광앉덱꾜 북걋 넝뮤 좋코, 혹컁슥 학귀 좋은 쿄쒼눼옳.   
2996  TRAIN_02996  방 쩐첵꺄 화위투 톤뉘랄 깖꿈학꼬 얩뿌눼옵! 뮤옅뵤따 빵예섟 꽝않뤼 헤슈옥쟝익 천뷰...   
2997  TRAIN_02997  줍번있 수싼학꽁 쉿섦윔 많임 낡깥댜. 림묘델륑한 쮜 열먀 앉 뙈셔 깖끎한 누뀜운 있...   
2998  TRAIN_02998  쿤 깊댑 않 한코 깠엊눈톄, 청먈 캐끗햐꼬 뮬됴 얾쩡 잚 낭욥꼭 꽁귀쩡청뀜웨 섕숟됴...   
2999  TRAIN_02999  1. 쭈찾쟝입 홈텔 뒷 뚜 뿔럭 찐낳아 있닭.   2. 플림밌염 찍쿄 팡잎 짝따. ...   

                                                 output  
0     별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자...  
1                  

In [18]:
samples = []

for i in range(10):
    sample = f"input : {train['input'][i]} \n output : {train['output'][i]}"
    samples.append(sample)

## 3) Model Load

In [19]:
# !pip install -U bitsandbytes
!pip show bitsandbytes

Name: bitsandbytes
Version: 0.48.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.10/dist-packages
Requires: numpy, packaging, torch
Required-by: 


In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델 ID
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(model_id)

# 모델 로드 (양자화 사용하지 않음)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    low_cpu_mem_usage=True  # 메모리 최적화 옵션
)

# GPU로 이동 (가능한 경우)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# # 테스트 입력
input_text = "안녕하세요! 오늘 기분은 어떠신가요?"
inputs = tokenizer(input_text, return_tensors="pt").to(device)

# # 모델 출력 생성
outputs = model.generate(inputs["input_ids"], max_length=50)
output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("모델 출력:", output_text)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


모델 출력: 안녕하세요! 오늘 기분은 어떠신가요? 편안한 마음으로 시작해보세요. 오늘의 목표는 무엇인가요? 어떤 도전을 겪고 싶으신가요?


In [26]:
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

## 4) Inference

In [74]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer
igimport torch
from tqdm import tqdm

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

restored_reviews = []

for index, row in tqdm(test.iterrows(), total=len(test)):
    query = row['input']

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant specializing in restoring obfuscated Korean reviews. "
                "Your task is to transform the given obfuscated Korean review into a clear, correct, "
                "and natural-sounding Korean review that reflects its original meaning. "
                "Below are examples of obfuscated Korean reviews and their restored forms:\n\n"
                f"Example, {samples}"
                "Spacing and word length in the output must be restored to the same as in the input. "
                "Do not provide any description. Print only in Korean."
            )
        },
        {
            "role": "user",
            "content": f"input : {query}, output : "
        },
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    print(f"\n--- {index+1} / {len(test)} 번째 복원 진행 중 ---")
    output_ids = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        streamer=streamer
    )
    print("\n--- 생성 완료 ---")

    result = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    restored_reviews.append(result)


  0%|          | 0/200 [00:00<?, ?it/s]


--- 1 / 200 번째 복원 진행 중 ---
녀물 머무는 게 마음에 듭니다. 정말 좋았어요. 콧수 좋네요.


  0%|          | 1/200 [00:01<05:03,  1.52s/it]


--- 생성 완료 ---

--- 2 / 200 번째 복원 진행 중 ---
풀립니다. 좀식을 업혀서 힘들어요. 위에 호텔 문화가 좋아요. 조심하세요.


  1%|          | 2/200 [00:03<05:22,  1.63s/it]

...


--- 생성 완료 ---

--- 199 / 200 번째 복원 진행 중 ---
맛있고 친근합니다.


100%|█████████▉| 199/200 [08:50<00:02,  2.34s/it]


--- 생성 완료 ---

--- 200 / 200 번째 복원 진행 중 ---
초히 오랜만이에요. 앞으로도 즐겁게 보내주세요.


100%|██████████| 200/200 [08:51<00:00,  2.66s/it]


--- 생성 완료 ---


## 5) Submission

In [27]:
submission = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/sample_submission.csv', encoding = 'utf-8')

In [82]:
print("len(submission) =", len(submission))
print("len(restored_reviews) =", len(restored_reviews))

len(submission) = 1689
len(restored_reviews) = 200


In [83]:
submission_li = submission.loc[:199].copy()

In [84]:
submission_li['output'] = restored_reviews[:200]

In [85]:
restored_reviews[:200]

["system\nYou are a helpful assistant specializing in restoring obfuscated Korean reviews. Your task is to transform the given obfuscated Korean review into a clear, correct, and natural-sounding Korean review that reflects its original meaning. Below are examples of obfuscated Korean reviews and their restored forms:\n\nExample, ['input : 별 한 게토 았깝땀. 왜 싸람듯릭 펼 1캐를 쥰눈징 컥꺾폰 싸람믐롯섞 맒록 섧멍핥쟈닐 탯끎룐눈 녀뮤 퀼교... 야뭍툰 둠 변 닺씨 깍낄 싫훈 굣. 깸삥읊 20여 년 댜녁뵨 곧 중 쩨윌 귑푼 낙팠떤 곶. \\n output : 별 한 개도 아깝다. 왜 사람들이 별 1개를 주는지 겪어본 사람으로서 말로 설명하자니 댓글로는 너무 길고... 아무튼 두 번 다시 가길 싫은 곳. 캠핑을 20여 년 다녀본 곳 중 제일 기분 나빴던 곳.', 'input : 잚많 쟉꼬 갉 태 좋눼욥. 차못동 줆 ㅋ \\n output : 잠만 자고 갈 때 좋네요. 잠옷도 줌 ㅋ', 'input : 절테 간면 않 된는 굣 멥몫 \\n output : 절대 가면 안 되는 곳 메모', 'input : 야... 칵컥 좋꾜 부됴 뼝 뚫렷썹 신원햐쥠만 닮패 넴센 밌쪄벅림. 샥퀘 핥류만 묵겠댜! 한눈 쌀람한뗌많 쭈쳔. 탐패 냄쌕갊 묘둔 쟝졈울 까저갼눈 콧. 놂랙팡엣셔 칵좋 닮패왕 윳흥예 천렸욹 택 냐눈 넴쌘갸 꼐쏙 방웨 잊슴 ㅆ... 샨닉깎 할 맑 엽숨. \\n output : 아... 가격 좋고 뷰도 뻥 뚫려서 시원하지만 담배 냄새 미쳐버림. 싸게 하루만 묵겠다! 하는 사람한테만 추천. 담배 냄새가 모든 장점을 가져가는 곳. 노래방에서 각종 담배와 유흥에 쩔었을 때 나는 냄새가 계속 방에 있음 ㅆ...

In [87]:
submission_li.to_csv('/kaggle/working/submission.csv', index = False, encoding = 'utf-8-sig')